# KEGG Pathway Analysis — Biological Interpretation
**Person 1 (Mila) — No GPU needed**

This notebook answers Research Question 3 from the project proposal:
> *'Кои молекуларни карактеристики (CpG локуси, гени) најмногу придонесуваат за разликување на различните типови карциноми?'*

What it does:
1. Finds the CpG probes that best separate the 6 cancer types (ANOVA)
2. Maps those probes to gene names using the Illumina 450K annotation
3. Runs KEGG pathway enrichment — finds which biological pathways are overrepresented
4. Saves a publication-ready bar chart of top pathways

**Runtime → Run all** — takes ~20 minutes

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q anndata numpy pandas scipy matplotlib seaborn gseapy

In [ ]:
SHARED_FOLDER_NAME = "multiomics-project"
from pathlib import Path
DRIVE_ROOT  = Path(f"/content/drive/MyDrive/{SHARED_FOLDER_NAME}")
METH_H5AD   = DRIVE_ROOT / "data/processed/tcga_methylation.h5ad"
RESULTS_DIR = DRIVE_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

N_TOP_PROBES = 500  # top discriminative CpG probes to use for pathway analysis

print('Methylation h5ad exists:', METH_H5AD.exists())

In [ ]:
import numpy as np
import pandas as pd
import anndata as ad
from scipy import stats

print('Loading methylation data...')
adata = ad.read_h5ad(METH_H5AD)
X = adata.X if isinstance(adata.X, np.ndarray) else adata.X.toarray()
labels = adata.obs['cancer_type'].values
probe_ids = adata.var.index.tolist()

print(f'Shape: {adata.shape}')
print(f'Cancer types: {dict(zip(*np.unique(labels, return_counts=True)))}')
print(f'Number of CpG probes: {len(probe_ids)}')

In [ ]:
# ANOVA: for each probe, test how different it is across 6 cancer types
# High F-statistic = this probe strongly separates cancer types
print('Running ANOVA across all probes (this takes ~5 minutes)...')

cancer_types = np.unique(labels)
f_stats = np.zeros(X.shape[1])

for i in range(X.shape[1]):
    groups = [X[labels == ct, i] for ct in cancer_types]
    f_stats[i], _ = stats.f_oneway(*groups)

# Replace NaN with 0
f_stats = np.nan_to_num(f_stats, nan=0.0)

# Top N most discriminative probes
top_indices = np.argsort(f_stats)[-N_TOP_PROBES:][::-1]
top_probe_ids = [probe_ids[i] for i in top_indices]
top_f_stats   = f_stats[top_indices]

print(f'Top probe F-statistic range: {top_f_stats[-1]:.1f} - {top_f_stats[0]:.1f}')
print(f'Example top probes: {top_probe_ids[:5]}')

In [ ]:
# Download Illumina 450K annotation — maps probe IDs to gene names
import urllib.request

ANNOTATION_URL = "https://raw.githubusercontent.com/chrisamiller/450k-annotation/master/HumanMethylation450_15017482_v1-2_filtered.csv"
ANNOTATION_PATH = Path("/content/450k_annotation.csv")

print('Downloading Illumina 450K annotation...')
try:
    urllib.request.urlretrieve(ANNOTATION_URL, ANNOTATION_PATH)
    annot = pd.read_csv(ANNOTATION_PATH, low_memory=False)
    print(f'Annotation loaded: {annot.shape}')
    print(f'Columns: {list(annot.columns[:10])}')
except Exception as e:
    print(f'Direct download failed ({e}), trying alternative...')
    # Alternative: use the UCSC annotation
    !wget -q -O /content/450k_annotation.csv.gz "https://zhouserver.research.chop.edu/InfiniumAnnotation/20180909/HM450/HM450.hg38.manifest.tsv.gz"
    annot = pd.read_csv('/content/450k_annotation.csv.gz', sep='\t', low_memory=False)
    print(f'Alternative annotation loaded: {annot.shape}')

In [ ]:
# Map probe IDs to gene names
# The annotation file has different possible column names — handle both
probe_col = None
gene_col  = None

for col in annot.columns:
    if 'probe' in col.lower() or col in ['IlmnID', 'Name', 'probeID']:
        probe_col = col
    if 'gene' in col.lower() and 'name' in col.lower():
        gene_col = col

if probe_col is None:
    probe_col = annot.columns[0]
if gene_col is None:
    # Try common column names
    for name in ['UCSC_RefGene_Name', 'gene', 'Gene_Name', 'geneNames']:
        if name in annot.columns:
            gene_col = name
            break

print(f'Using probe column: {probe_col}')
print(f'Using gene column : {gene_col}')

annot_clean = annot[[probe_col, gene_col]].copy()
annot_clean.columns = ['probe_id', 'gene_name']
annot_clean = annot_clean.dropna(subset=['gene_name'])
annot_clean = annot_clean[annot_clean['gene_name'] != '']
probe_to_gene = dict(zip(annot_clean['probe_id'], annot_clean['gene_name']))
print(f'Probes with gene annotation: {len(probe_to_gene)}')

In [ ]:
# Get gene list from top probes
gene_list = []
probe_gene_pairs = []

for probe_id, f_stat in zip(top_probe_ids, top_f_stats):
    raw_gene = probe_to_gene.get(probe_id, '')
    if not raw_gene:
        continue
    # Some probes map to multiple genes separated by ';'
    for gene in str(raw_gene).split(';'):
        gene = gene.strip()
        if gene and gene != 'NA':
            gene_list.append(gene)
            probe_gene_pairs.append({'probe_id': probe_id, 'gene': gene, 'f_statistic': f_stat})

gene_list = list(set(gene_list))  # unique genes

# Save top probes table
df_probes = pd.DataFrame(probe_gene_pairs)
df_probes.to_csv(RESULTS_DIR / 'top_discriminative_probes.csv', index=False)

print(f'Unique genes from top {N_TOP_PROBES} probes: {len(gene_list)}')
print(f'Example genes: {gene_list[:10]}')

In [ ]:
# Run KEGG pathway enrichment
import gseapy as gp

print('Running KEGG enrichment...')
enr = gp.enrichr(
    gene_list=gene_list,
    gene_sets='KEGG_2021_Human',
    organism='human',
    outdir=None,
    cutoff=0.05
)

results = enr.results.sort_values('Adjusted P-value').head(20)
results.to_csv(RESULTS_DIR / 'kegg_results.csv', index=False)

print(f'Top enriched pathways:')
for _, row in results.head(10).iterrows():
    print(f"  {row['Term'][:60]:<60} p={row['Adjusted P-value']:.4f}")

In [ ]:
# Publication-ready KEGG bar chart
import matplotlib.pyplot as plt
import numpy as np

top20 = results.head(20).copy()
top20['-log10(padj)'] = -np.log10(top20['Adjusted P-value'].clip(1e-10))
top20 = top20.sort_values('-log10(padj)')

# Clean pathway names (remove 'KEGG_' prefix if present)
top20['Term'] = top20['Term'].str.replace(r'^KEGG_', '', regex=True).str.replace('_', ' ')

fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(
    top20['Term'],
    top20['-log10(padj)'],
    color=plt.cm.RdYlBu_r(np.linspace(0.2, 0.8, len(top20))),
    edgecolor='white', linewidth=0.5
)
ax.axvline(x=-np.log10(0.05), color='red', linestyle='--', linewidth=1, alpha=0.7, label='p=0.05')
ax.set_xlabel('-log₁₀(Adjusted P-value)', fontsize=12)
ax.set_title(
    f'KEGG Pathway Enrichment\nTop {N_TOP_PROBES} Discriminative CpG Probes across 6 Cancer Types',
    fontsize=13, fontweight='bold'
)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'kegg_top_pathways.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to Drive!')

In [ ]:
# Also: show per-cancer-type methylation heatmap for top 30 probes
import seaborn as sns

top30_idx   = top_indices[:30]
top30_ids   = [probe_ids[i] for i in top30_idx]
X_top30     = X[:, top30_idx]

# Mean beta value per cancer type per probe
heatmap_data = pd.DataFrame(
    {ct: X_top30[labels == ct].mean(axis=0) for ct in np.unique(labels)},
    index=[f'{p[:10]}...' if len(p) > 10 else p for p in top30_ids]
)

fig, ax = plt.subplots(figsize=(10, 10))
sns.heatmap(
    heatmap_data,
    cmap='RdBu_r', center=0.5, vmin=0, vmax=1,
    ax=ax, linewidths=0.3,
    cbar_kws={'label': 'Mean Beta Value (methylation level)'}
)
ax.set_title('Top 30 Discriminative CpG Probes\nMean Methylation per Cancer Type',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Cancer Type', fontsize=11)
ax.set_ylabel('CpG Probe', fontsize=11)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'methylation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nAll done! Files saved to Drive/results/:')
print('  kegg_top_pathways.png       — share with group')
print('  methylation_heatmap.png     — share with group')
print('  kegg_results.csv            — full enrichment table')
print('  top_discriminative_probes.csv — top 500 probes with gene names')